# CLEAN Predictor Baselines On Metalloenzyme EC Splits

This notebook prepares and optionally trains the original CLEAN sequence-based EC predictor on the metalloenzyme EC test sets used for the DeepMzyme comparison.

It supports the planned CLEAN comparison matrix:

- `clean30_<source>_fold{fold}_metallo`: train CLEAN on the selected extracted CLEAN30 metalloenzyme source for that fold, test only on the selected CLEAN30 metalloenzyme test proteins from the same fold.
- `clean30_<source>_fold{fold}_full`: train CLEAN on the full original CLEAN30 train split that the metalloenzymes were extracted from, test only on the selected CLEAN30 metalloenzyme test proteins from the same fold.
- `care30_metallo`: train CLEAN on extracted CARE Task 1 clusterRes30 metalloenzymes, test only on the extracted CARE30 metalloenzyme test proteins.
- `care30_full`: train CLEAN on the full original CARE Task 1 train set, test only on the extracted CARE30 metalloenzyme test proteins.

By default, the notebook prepares all available CLEAN30 folds 0-4 from the conservative `single_donor_supported_metal_conservative` CLEAN source and CARE30, with both `metallo` and `full` training scopes. In the main DeepMzyme notebook, `CLEAN_SHARED_SOURCE = "main"` means this same conservative single-donor supported-metal source. This CLEAN baseline notebook accepts `CLEAN_METALLO_SOURCE = "main"` as the same alias and normalizes it to `single_donor_supported_metal_conservative`. The data source can be local, Drive, uploaded, or the separate CLEAN-only HuggingFace bundle. That bundle contains only sequence/split CSVs and metalloenzyme summary CSVs; it deliberately does not include DeepMzyme structures, ESMC embeddings, RING files, or graph features.

The notebook defaults to table preparation only. Official CLEAN installation, ESM-1b embedding generation, CLEAN training, inference, and scoring are long steps controlled by explicit flags in the config cell.


In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import urllib.request
from typing import Iterable

import pandas as pd


SUMMARY_CSV_NAME = "final_data_summarazing_table_transition_metals_only_catalytic.csv"


def find_project_root(start: Path | None = None) -> Path:
    cur = (start or Path.cwd()).resolve()
    for path in [cur, *cur.parents]:
        if (path / "Plan.md").exists() and (path / "DeepMzyme_Data").exists():
            return path
    for path in [cur, *cur.parents]:
        if (path / "Plan.md").exists():
            return path
    raise RuntimeError("Could not find DeepMzyme project root from current working directory")


PROJECT_ROOT = find_project_root()
WORK_ROOT = PROJECT_ROOT / "CLEAN" / "work"
PREPARED_ROOT = WORK_ROOT / "prepared_tables"
RESULTS_ROOT = WORK_ROOT / "scored_results"
OFFICIAL_CLEAN_DIR = WORK_ROOT / "official_CLEAN"
OFFICIAL_CLEAN_APP = OFFICIAL_CLEAN_DIR / "app"
CLEAN_REPO_URL = "https://github.com/tttianhao/CLEAN.git"

# ── Data source ───────────────────────────────────────────────────────────────
# auto uses local DeepMzyme_Data when all selected sources are present; otherwise it downloads the CLEAN-only bundle.
CLEAN_DATA_SOURCE = "auto"  #@param ["auto", "local_project", "huggingface_link", "upload_file", "drive"]
CLEAN_DRIVE_DATA_ROOT = "/content/drive/MyDrive/DeepMzyme/DeepMzyme_Data"  #@param {type:"string"}
CLEAN_DATA_ROOT_OVERRIDE = ""  #@param {type:"string"}
CLEAN_BUNDLE_FILENAME = "CLEAN_predictor_baselines_v2_clean30x5_single_donor_supported_metal_conservative_care30_sources.tar.zst"
CLEAN_BUNDLE_URL = "https://huggingface.co/datasets/GMBioinformatics/DeepMzyme/resolve/main/CLEAN_predictor_baselines_v2_clean30x5_single_donor_supported_metal_conservative_care30_sources.tar.zst"
CLEAN_BUNDLE_SHA256 = "5124b0b514b49affc158df121a87f5389ec1e027d14e0cf0a53cfb13a602c0f0"
CLEAN_BUNDLE_EXTRACT_ROOT = ""  # blank = CLEAN/work/bundle_data

# ── Job matrix ────────────────────────────────────────────────────────────────
# CLEAN30 means the five extracted CLEAN_30 metalloenzyme folds currently available.
# The conservative source selects one AlphaFill donor per UniProt target and keeps the supported-metal rows from that donor.
# "main" is the same source as "single_donor_supported_metal_conservative", matching the main DeepMzyme notebook.
CLEAN_METALLO_SOURCE = "main"  #@param ["main", "single_donor_supported_metal_conservative", "shared"]
CLEAN_BENCHMARKS_CSV = "clean30,care30"  #@param {type:"string"}
CLEAN_FOLDS_CSV = "0,1,2,3,4"  #@param {type:"string"}
TRAIN_SCOPES_CSV = "metallo,full"  #@param {type:"string"}
CLEAR_PREPARED_TABLES = True  #@param {type:"boolean"}

# Long-step flags. Keep these False until the normalized tables look correct.
RUN_INSTALL_CLEAN = False  #@param {type:"boolean"}
INSTALL_REQUIREMENTS = False  #@param {type:"boolean"}
RUN_GENERATE_ESM_AND_DISTANCES = False  #@param {type:"boolean"}
RUN_TRAIN_CLEAN = False  #@param {type:"boolean"}
RUN_INFERENCE = False  #@param {type:"boolean"}
RUN_SCORE_RESULTS = False  #@param {type:"boolean"}

# Do not use pre-MAHOMES CARE candidate sites as final metalloenzyme data unless explicitly debugging.
ALLOW_PROVISIONAL_CARE_INPUTS = False

# Official CLEAN recommends 2500 epochs for split30 triplet training. Reduce only for smoke tests.
TRIPLET_EPOCHS = 2500
TRIPLET_LR = 5e-4
PYTHON = sys.executable


def parse_csv_tokens(value: str, *, allowed: set[str] | None = None) -> list[str]:
    tokens = []
    for item in str(value).split(","):
        token = item.strip().lower()
        if not token:
            continue
        if allowed is not None and token not in allowed:
            raise ValueError(f"Unsupported value {token!r}; allowed values are {sorted(allowed)}")
        if token not in tokens:
            tokens.append(token)
    if not tokens:
        raise ValueError("CSV selector produced no values")
    return tokens


def parse_csv_ints(value: str) -> list[int]:
    values = []
    for item in str(value).split(","):
        item = item.strip()
        if not item:
            continue
        values.append(int(item))
    if not values:
        raise ValueError("CSV integer selector produced no values")
    return values


SELECTED_BENCHMARKS = parse_csv_tokens(CLEAN_BENCHMARKS_CSV, allowed={"clean30", "care30"})
SELECTED_CLEAN_IDENTITIES = sorted({int(token.replace("clean", "")) for token in SELECTED_BENCHMARKS if token.startswith("clean")})
SELECTED_CLEAN_FOLDS = parse_csv_ints(CLEAN_FOLDS_CSV)
SELECTED_TRAIN_SCOPES = parse_csv_tokens(TRAIN_SCOPES_CSV, allowed={"metallo", "full"})


CLEAN_METALLO_SOURCE_ROOTS = {
    "shared": "CLEAN_{identity}_shared",
    "single_donor_supported_metal_conservative": "CLEAN_{identity}_shared_single_donor_supported_metal_conservative",
}
CLEAN_METALLO_SOURCE_TAGS = {
    "shared": "shared",
    "single_donor_supported_metal_conservative": "sdsmc",
}


def normalize_clean_metallo_source(value: str) -> str:
    key = str(value).strip().lower()
    aliases = {
        "original": "shared",
        "multi_donor": "shared",
        "main": "single_donor_supported_metal_conservative",
        "conservative": "single_donor_supported_metal_conservative",
        "single_donor": "single_donor_supported_metal_conservative",
        "single_donor_conservative": "single_donor_supported_metal_conservative",
    }
    key = aliases.get(key, key)
    if key not in CLEAN_METALLO_SOURCE_ROOTS:
        raise ValueError(f"Unsupported CLEAN_METALLO_SOURCE={value!r}; allowed values are {sorted(CLEAN_METALLO_SOURCE_ROOTS)}")
    return key


SELECTED_CLEAN_METALLO_SOURCE = normalize_clean_metallo_source(CLEAN_METALLO_SOURCE)
CLEAN_METALLO_SOURCE_TAG = CLEAN_METALLO_SOURCE_TAGS[SELECTED_CLEAN_METALLO_SOURCE]


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_sha256(path: Path, expected: str) -> None:
    expected = str(expected or "").strip().lower()
    if not expected:
        print("No SHA256 provided; skipping checksum verification for:", path)
        return
    observed = sha256_file(path).lower()
    if observed != expected:
        raise ValueError(f"SHA256 mismatch for {path}: expected {expected}, observed {observed}")
    print("Verified CLEAN bundle SHA256:", observed)


def ensure_zstd_available() -> None:
    if shutil.which("zstd") is not None:
        return
    if Path("/content").exists():
        subprocess.run(["apt-get", "update"], check=True)
        subprocess.run(["apt-get", "install", "-y", "zstd"], check=True)
        return
    raise RuntimeError("zstd is required to unpack .tar.zst bundles but was not found on PATH.")


def unpack_bundle(bundle_path: Path, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    marker = destination / ".clean_predictor_bundle_unpacked"
    stat = bundle_path.stat()
    identity = f"{bundle_path.resolve()}|{stat.st_size}|{stat.st_mtime_ns}"
    if marker.exists() and marker.read_text(encoding="utf-8").strip() == identity:
        print("CLEAN source bundle already unpacked:", destination)
        return
    if any(destination.iterdir()):
        shutil.rmtree(destination)
        destination.mkdir(parents=True, exist_ok=True)
    ensure_zstd_available()
    subprocess.run(["tar", "--use-compress-program=zstd", "-xf", str(bundle_path), "-C", str(destination)], check=True)
    marker.write_text(identity + "\n", encoding="utf-8")
    print("Unpacked CLEAN source bundle to:", destination)


def clean_full_train_csv(data_root: Path, identity: int, fold: int) -> Path:
    return Path(data_root) / "CLEAN_all_train_valid_splits" / f"split{identity}" / f"split{identity}_train_split_{fold}.csv"


def clean_full_test_csv(data_root: Path, identity: int, fold: int) -> Path:
    return Path(data_root) / "CLEAN_all_train_valid_splits" / f"split{identity}" / f"split{identity}_test_split_{fold}_curate.csv"


def clean_metallo_summary_csv(data_root: Path, identity: int, fold: int, split: str) -> Path:
    data_root = Path(data_root)
    root_name = CLEAN_METALLO_SOURCE_ROOTS[SELECTED_CLEAN_METALLO_SOURCE].format(identity=identity)
    shared = data_root / root_name / "folds" / f"CLEAN_{identity}_train_test_split_{fold}_{split}.csv"
    if shared.exists():
        return shared
    direct = data_root / f"CLEAN_{identity}_train_test_split_{fold}" / split / SUMMARY_CSV_NAME
    return direct


def care_full_train_csv(data_root: Path) -> Path:
    return Path(data_root) / "CARE_dataset" / "CARE_datasets" / "splits" / "task1" / "protein_train.csv"


def care_full_test_csv(data_root: Path) -> Path:
    return Path(data_root) / "CARE_dataset" / "CARE_datasets" / "splits" / "task1" / "30_protein_test.csv"


def care_metallo_summary_csv(data_root: Path, split: str) -> Path:
    return Path(data_root) / "CARE_task1_30_clusterRes30_train_test_metallo" / split / SUMMARY_CSV_NAME


def selected_sources_available(data_root: Path, *, verbose: bool = False) -> bool:
    required: list[Path] = []
    for identity in SELECTED_CLEAN_IDENTITIES:
        for fold in SELECTED_CLEAN_FOLDS:
            required.extend([
                clean_full_train_csv(data_root, identity, fold),
                clean_full_test_csv(data_root, identity, fold),
                clean_metallo_summary_csv(data_root, identity, fold, "train"),
                clean_metallo_summary_csv(data_root, identity, fold, "test"),
            ])
    if "care30" in SELECTED_BENCHMARKS:
        required.extend([
            care_full_train_csv(data_root),
            care_full_test_csv(data_root),
            care_metallo_summary_csv(data_root, "train"),
            care_metallo_summary_csv(data_root, "test"),
        ])
    missing = [path for path in required if not path.exists()]
    if verbose and missing:
        print("Missing selected CLEAN source files:")
        for path in missing[:20]:
            print("  ", path)
        if len(missing) > 20:
            print(f"  ... {len(missing) - 20} more")
    return not missing


def resolve_clean_data_root() -> Path:
    if CLEAN_DATA_ROOT_OVERRIDE.strip():
        root = Path(CLEAN_DATA_ROOT_OVERRIDE).expanduser().resolve()
        if not selected_sources_available(root, verbose=True):
            raise FileNotFoundError(f"CLEAN_DATA_ROOT_OVERRIDE does not contain the selected CLEAN source files: {root}")
        return root

    local_root = PROJECT_ROOT / "DeepMzyme_Data"
    source = str(CLEAN_DATA_SOURCE).strip().lower()
    if source == "auto" and selected_sources_available(local_root):
        print("Using local project DeepMzyme_Data for CLEAN sources:", local_root)
        return local_root
    if source == "local_project":
        if not selected_sources_available(local_root, verbose=True):
            raise FileNotFoundError(f"Local project DeepMzyme_Data does not contain the selected CLEAN source files: {local_root}")
        return local_root
    if source == "drive":
        if Path("/content").exists():
            from google.colab import drive
            drive.mount("/content/drive")
        root = Path(CLEAN_DRIVE_DATA_ROOT).expanduser().resolve()
        if not selected_sources_available(root, verbose=True):
            raise FileNotFoundError(f"Drive data root does not contain the selected CLEAN source files: {root}")
        return root
    if source == "upload_file":
        if not Path("/content").exists():
            raise RuntimeError("upload_file mode is available only in Colab.")
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No CLEAN source bundle uploaded.")
        bundle_path = Path(next(iter(uploaded.keys()))).resolve()
    elif source in {"auto", "huggingface_link"}:
        if not CLEAN_BUNDLE_URL.strip():
            raise ValueError("CLEAN_BUNDLE_URL is empty; cannot download the CLEAN source bundle.")
        bundle_dir = WORK_ROOT / "bundles"
        bundle_dir.mkdir(parents=True, exist_ok=True)
        bundle_path = bundle_dir / CLEAN_BUNDLE_FILENAME
        if not bundle_path.exists():
            print("Downloading CLEAN source bundle:", CLEAN_BUNDLE_URL)
            urllib.request.urlretrieve(CLEAN_BUNDLE_URL, bundle_path)
        else:
            print("Using existing CLEAN source bundle:", bundle_path)
        verify_sha256(bundle_path, CLEAN_BUNDLE_SHA256)
    else:
        raise ValueError(f"Unsupported CLEAN_DATA_SOURCE={CLEAN_DATA_SOURCE!r}")

    extract_root = Path(CLEAN_BUNDLE_EXTRACT_ROOT).expanduser() if CLEAN_BUNDLE_EXTRACT_ROOT.strip() else WORK_ROOT / "bundle_data"
    extract_root = extract_root.resolve()
    unpack_bundle(bundle_path, extract_root)
    root = extract_root / "DeepMzyme_Data"
    if not selected_sources_available(root, verbose=True):
        raise FileNotFoundError(f"Unpacked CLEAN source bundle does not contain the selected source files: {root}")
    return root


for directory in [WORK_ROOT, PREPARED_ROOT, RESULTS_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)

CLEAN_DATA_ROOT = resolve_clean_data_root()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("WORK_ROOT:", WORK_ROOT)
print("CLEAN_DATA_ROOT:", CLEAN_DATA_ROOT)
print("OFFICIAL_CLEAN_APP:", OFFICIAL_CLEAN_APP)
print("SELECTED_BENCHMARKS:", SELECTED_BENCHMARKS)
print("SELECTED_CLEAN_FOLDS:", SELECTED_CLEAN_FOLDS)
print("SELECTED_TRAIN_SCOPES:", SELECTED_TRAIN_SCOPES)
print("SELECTED_CLEAN_METALLO_SOURCE:", SELECTED_CLEAN_METALLO_SOURCE)


## Data Normalization Helpers

Official CLEAN expects tab-delimited files named `data/<name>.csv` with columns:

```text
Entry    EC number    Sequence
```

The helpers below normalize the source CLEAN/CARE files and aggregate duplicate protein rows into one semicolon-separated EC label field per protein.

In [ ]:
EC_SPLIT_RE = re.compile(r"[;,]")
VALID_AA_RE = re.compile(r"^[A-Z*<>-]+$")


def read_table(path: Path) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    # CLEAN source files are TSV with .csv extension; CARE and MAHOMES summaries are CSV.
    with path.open("r", encoding="utf-8", errors="ignore") as handle:
        first = handle.readline()
    sep = "\t" if "\t" in first and first.count("\t") >= first.count(",") else ","
    df = pd.read_csv(path, sep=sep, dtype=str, keep_default_na=False)
    unnamed = [c for c in df.columns if c.startswith("Unnamed") or c == ""]
    if unnamed:
        df = df.drop(columns=unnamed)
    return df


def first_existing_column(df: pd.DataFrame, candidates: Iterable[str]) -> str:
    lower = {c.lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate in df.columns:
            return candidate
        if candidate.lower() in lower:
            return lower[candidate.lower()]
    raise KeyError(f"None of the candidate columns exist: {list(candidates)}; columns={list(df.columns)}")


def split_ecs(value: str) -> list[str]:
    ecs = []
    for item in EC_SPLIT_RE.split(str(value)):
        item = item.strip()
        if not item or item.lower() in {"nan", "none"}:
            continue
        ecs.append(item)
    return sorted(set(ecs))


def clean_sequence(seq: str) -> str:
    seq = str(seq).strip().replace(" ", "").replace("\n", "")
    return seq


def aggregate_rows(rows: Iterable[dict], *, context: str) -> pd.DataFrame:
    by_entry: dict[str, dict] = {}
    missing_sequence = []
    for row in rows:
        entry = str(row["Entry"]).strip()
        seq = clean_sequence(row.get("Sequence", ""))
        ecs = split_ecs(row.get("EC number", ""))
        if not entry or not ecs:
            continue
        if not seq:
            missing_sequence.append(entry)
            continue
        rec = by_entry.setdefault(entry, {"Entry": entry, "ECs": set(), "Sequence": seq})
        if rec["Sequence"] != seq:
            # Keep the first sequence; source split IDs should not have conflicting sequences.
            pass
        rec["ECs"].update(ecs)
    if missing_sequence:
        preview = ", ".join(sorted(set(missing_sequence))[:10])
        raise ValueError(f"{context}: missing sequence for {len(set(missing_sequence))} entries, e.g. {preview}")
    out = pd.DataFrame(
        {
            "Entry": rec["Entry"],
            "EC number": ";".join(sorted(rec["ECs"])),
            "Sequence": rec["Sequence"],
        }
        for rec in by_entry.values()
    )
    if out.empty:
        raise ValueError(f"{context}: no rows after normalization")
    return out.sort_values("Entry").reset_index(drop=True)


def normalize_sequence_source(path: Path, *, context: str) -> pd.DataFrame:
    df = read_table(path)
    entry_col = first_existing_column(df, ["Entry", "ID", "protein_id", "uniprot_id"])
    ec_col = first_existing_column(df, ["EC number", "EC", "EC All", "ecnumber"])
    seq_col = first_existing_column(df, ["Sequence", "Sequences", "sequence"])
    rows = (
        {"Entry": row[entry_col], "EC number": row[ec_col], "Sequence": row[seq_col]}
        for _, row in df.iterrows()
    )
    return aggregate_rows(rows, context=context)


def build_sequence_map(*source_paths: Path) -> dict[str, str]:
    seqs = {}
    for source in source_paths:
        if not Path(source).exists():
            continue
        df = normalize_sequence_source(source, context=f"sequence_map:{source.name}")
        seqs.update(dict(zip(df["Entry"], df["Sequence"])))
    return seqs


def normalize_metallo_summary(path: Path, *, sequence_sources: list[Path], context: str) -> pd.DataFrame:
    df = read_table(path)
    entry_col = first_existing_column(df, ["uniprot_id", "protein_id", "Entry", "ID"])
    ec_col = first_existing_column(df, ["ecnumber", "EC number", "EC", "EC All"])
    seq_map = build_sequence_map(*sequence_sources)
    rows = []
    missing = []
    for _, row in df.iterrows():
        entry = str(row[entry_col]).strip()
        seq = seq_map.get(entry, "")
        if not seq:
            missing.append(entry)
        rows.append({"Entry": entry, "EC number": row[ec_col], "Sequence": seq})
    if missing:
        preview = ", ".join(sorted(set(missing))[:10])
        raise ValueError(f"{context}: {len(set(missing))} entries not found in sequence sources, e.g. {preview}")
    return aggregate_rows(rows, context=context)


def write_clean_table(df: pd.DataFrame, name: str) -> Path:
    path = PREPARED_ROOT / f"{name}.csv"
    df[["Entry", "EC number", "Sequence"]].to_csv(path, sep="\t", index=False)
    return path


def summarize_clean_table(df: pd.DataFrame, name: str) -> dict:
    ec_lists = df["EC number"].map(split_ecs)
    ec1 = ec_lists.map(lambda xs: sorted({x.split(".")[0] for x in xs if x})).explode().value_counts().sort_index().to_dict()
    ec2 = ec_lists.map(lambda xs: sorted({".".join(x.split(".")[:2]) for x in xs if len(x.split(".")) >= 2})).explode().value_counts().sort_index().to_dict()
    return {
        "name": name,
        "proteins": int(len(df)),
        "unique_ec": int(len(set(e for xs in ec_lists for e in xs))),
        "ec1_counts": ec1,
        "ec2_classes": int(len(ec2)),
    }

## Prepare The CLEAN Predictor Job Matrix

This cell creates the selected benchmark x train-scope x fold jobs. With defaults, it prepares CLEAN30 folds 0-4 using both `metallo` and `full` training scopes, plus CARE30 using both scopes.

Every job tests only on the extracted metalloenzyme test subset for that benchmark/fold. Full-training jobs use the original full training split that the metalloenzyme subset was extracted from; metallo-training jobs use only the extracted metalloenzyme train subset.


In [ ]:
if CLEAR_PREPARED_TABLES and PREPARED_ROOT.exists():
    shutil.rmtree(PREPARED_ROOT)
PREPARED_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

prepared = {}
summaries = []
jobs = []
source_status = {}


def remember_table(name: str, df: pd.DataFrame) -> None:
    prepared[name] = write_clean_table(df, name)
    summaries.append(summarize_clean_table(df, name))


def add_job(
    *,
    job_name: str,
    train_data: str,
    test_source_data: str,
    test_data: str,
    model_name: str,
    benchmark: str,
    train_scope: str,
    test_scope: str = "metalloenzyme_only",
    clean_identity: int | None = None,
    clean_fold: int | None = None,
    clean_metallo_source: str = "",
) -> None:
    jobs.append({
        "job_name": job_name,
        "train_data": train_data,
        "test_source_data": test_source_data,
        "test_data": test_data,
        "model_name": model_name,
        "benchmark": benchmark,
        "train_scope": train_scope,
        "test_scope": test_scope,
        "clean_identity": "" if clean_identity is None else int(clean_identity),
        "clean_fold": "" if clean_fold is None else int(clean_fold),
        "clean_metallo_source": clean_metallo_source,
    })


# CLEAN identity/fold jobs. Current bundle supports CLEAN30 folds 0-4.
for identity in SELECTED_CLEAN_IDENTITIES:
    clean_source_tag = CLEAN_METALLO_SOURCE_TAG
    clean_benchmark = f"CLEAN{identity}_{clean_source_tag}"
    for fold in SELECTED_CLEAN_FOLDS:
        clean_full_train = clean_full_train_csv(CLEAN_DATA_ROOT, identity, fold)
        clean_full_test = clean_full_test_csv(CLEAN_DATA_ROOT, identity, fold)
        clean_metallo_train = clean_metallo_summary_csv(CLEAN_DATA_ROOT, identity, fold, "train")
        clean_metallo_test = clean_metallo_summary_csv(CLEAN_DATA_ROOT, identity, fold, "test")

        clean_prefix = f"clean{identity}_{clean_source_tag}_fold{fold}"
        clean_full_train_name = f"{clean_prefix}_full_train"
        clean_metallo_train_name = f"{clean_prefix}_metallo_train"
        clean_metallo_test_name = f"{clean_prefix}_metallo_test"

        clean_full_train_df = normalize_sequence_source(clean_full_train, context=clean_full_train_name)
        clean_metallo_train_df = normalize_metallo_summary(
            clean_metallo_train,
            sequence_sources=[clean_full_train, clean_full_test],
            context=clean_metallo_train_name,
        )
        clean_metallo_test_df = normalize_metallo_summary(
            clean_metallo_test,
            sequence_sources=[clean_full_test, clean_full_train],
            context=clean_metallo_test_name,
        )
        remember_table(clean_full_train_name, clean_full_train_df)
        remember_table(clean_metallo_train_name, clean_metallo_train_df)
        remember_table(clean_metallo_test_name, clean_metallo_test_df)

        if "metallo" in SELECTED_TRAIN_SCOPES:
            add_job(
                job_name=f"{clean_prefix}_metallo",
                train_data=clean_metallo_train_name,
                test_source_data=clean_metallo_test_name,
                test_data=f"{clean_prefix}_metallo_test__for_metallo_train",
                model_name=f"{clean_prefix}_metallo_triplet",
                benchmark=clean_benchmark,
                train_scope="metalloenzyme_only",
                clean_identity=identity,
                clean_fold=fold,
                clean_metallo_source=SELECTED_CLEAN_METALLO_SOURCE,
            )
        if "full" in SELECTED_TRAIN_SCOPES:
            add_job(
                job_name=f"{clean_prefix}_full",
                train_data=clean_full_train_name,
                test_source_data=clean_metallo_test_name,
                test_data=f"{clean_prefix}_metallo_test__for_full_train",
                model_name=f"{clean_prefix}_full_triplet",
                benchmark=clean_benchmark,
                train_scope="full_original_train_split",
                clean_identity=identity,
                clean_fold=fold,
                clean_metallo_source=SELECTED_CLEAN_METALLO_SOURCE,
            )
        source_status[f"{clean_prefix}"] = f"final_mahomes_export:{SELECTED_CLEAN_METALLO_SOURCE}"

# CARE Task 1 clusterRes30 jobs.
if "care30" in SELECTED_BENCHMARKS:
    care_full_train = care_full_train_csv(CLEAN_DATA_ROOT)
    care_full_test = care_full_test_csv(CLEAN_DATA_ROOT)
    care_metallo_train_final = care_metallo_summary_csv(CLEAN_DATA_ROOT, "train")
    care_metallo_test_final = care_metallo_summary_csv(CLEAN_DATA_ROOT, "test")
    care_metallo_train_provisional = Path("/media/Data/care_sets/task1_30_clusterRes30/mahomes_inputs/train/candidate_site_summary.csv")
    care_metallo_test_provisional = Path("/media/Data/care_sets/task1_30_clusterRes30/mahomes_inputs/test/candidate_site_summary.csv")

    care_metallo_ready = care_metallo_train_final.exists() and care_metallo_test_final.exists()
    care_metallo_source_status = "final_mahomes_export" if care_metallo_ready else "missing_final_mahomes_export"
    if not care_metallo_ready and ALLOW_PROVISIONAL_CARE_INPUTS:
        care_metallo_ready = care_metallo_train_provisional.exists() and care_metallo_test_provisional.exists()
        care_metallo_source_status = "PROVISIONAL_pre_mahomes_candidate_sites" if care_metallo_ready else care_metallo_source_status
        care_metallo_train_path = care_metallo_train_provisional
        care_metallo_test_path = care_metallo_test_provisional
    else:
        care_metallo_train_path = care_metallo_train_final
        care_metallo_test_path = care_metallo_test_final

    if not care_metallo_ready:
        raise FileNotFoundError(
            "CARE30 selected but final CARE metalloenzyme summaries are missing. "
            f"Expected train={care_metallo_train_final}, test={care_metallo_test_final}."
        )

    care30_full_train_name = "care30_full_train"
    care30_metallo_train_name = "care30_clusterRes30_metallo_train"
    care30_metallo_test_name = "care30_clusterRes30_metallo_test"

    care30_full_train_df = normalize_sequence_source(care_full_train, context=care30_full_train_name)
    care30_metallo_train_df = normalize_metallo_summary(
        care_metallo_train_path,
        sequence_sources=[care_full_train, care_full_test],
        context=care30_metallo_train_name,
    )
    care30_metallo_test_df = normalize_metallo_summary(
        care_metallo_test_path,
        sequence_sources=[care_full_test, care_full_train],
        context=care30_metallo_test_name,
    )
    remember_table(care30_full_train_name, care30_full_train_df)
    remember_table(care30_metallo_train_name, care30_metallo_train_df)
    remember_table(care30_metallo_test_name, care30_metallo_test_df)

    if "metallo" in SELECTED_TRAIN_SCOPES:
        add_job(
            job_name="care30_metallo",
            train_data=care30_metallo_train_name,
            test_source_data=care30_metallo_test_name,
            test_data="care30_clusterRes30_metallo_test__for_metallo_train",
            model_name="care30_metallo_triplet",
            benchmark="CARE30",
            train_scope="metalloenzyme_only_clusterRes30",
        )
    if "full" in SELECTED_TRAIN_SCOPES:
        add_job(
            job_name="care30_full",
            train_data=care30_full_train_name,
            test_source_data=care30_metallo_test_name,
            test_data="care30_clusterRes30_metallo_test__for_full_train",
            model_name="care30_full_triplet",
            benchmark="CARE30",
            train_scope="full_original_task1_train",
        )
    source_status["care30"] = care_metallo_source_status

if not jobs:
    raise RuntimeError("No CLEAN jobs were generated. Check CLEAN_BENCHMARKS_CSV and TRAIN_SCOPES_CSV.")

# Duplicate test tables under run-specific names to avoid official CLEAN result overwrite.
for job in jobs:
    source_path = prepared[job["test_source_data"]]
    target_path = PREPARED_ROOT / f"{job['test_data']}.csv"
    shutil.copy2(source_path, target_path)
    prepared[job["test_data"]] = target_path

manifest = pd.DataFrame(jobs)
manifest["source_status_json"] = json.dumps(source_status, sort_keys=True)
manifest["clean_data_root"] = str(CLEAN_DATA_ROOT)
manifest["selected_clean_metallo_source"] = SELECTED_CLEAN_METALLO_SOURCE
manifest["bundle_filename"] = CLEAN_BUNDLE_FILENAME if CLEAN_DATA_SOURCE in {"auto", "huggingface_link", "upload_file"} else ""
manifest_path = PREPARED_ROOT / "jobs_manifest.csv"
summary_path = PREPARED_ROOT / "table_summary.json"
manifest.to_csv(manifest_path, index=False)
summary_path.write_text(json.dumps(summaries, indent=2, sort_keys=True), encoding="utf-8")

print("Prepared tables:")
for name, table_path in sorted(prepared.items()):
    print(f"  {name}: {table_path}")
print("\nJob manifest:", manifest_path)
try:
    display(manifest)
except NameError:
    print(manifest.to_string(index=False))
print("\nTable summaries:")
print(summary_path.read_text())


## Install Official CLEAN Code

This clones the official CLEAN implementation into `CLEAN/work/official_CLEAN` and installs it from its `app/` directory.

Run this only in the environment where you want to train CLEAN. Official CLEAN was developed with Python 3.10 and ESM-1b; dependency conflicts with the DeepMzyme environment are possible, so a separate environment is preferable for serious CLEAN runs.

In [ ]:
if RUN_INSTALL_CLEAN:
    if not OFFICIAL_CLEAN_DIR.exists():
        subprocess.run(["git", "clone", CLEAN_REPO_URL, str(OFFICIAL_CLEAN_DIR)], check=True)
    if INSTALL_REQUIREMENTS:
        subprocess.run([PYTHON, "-m", "pip", "install", "-r", "requirements.txt"], cwd=OFFICIAL_CLEAN_APP, check=True)
    subprocess.run([PYTHON, "build.py", "install"], cwd=OFFICIAL_CLEAN_APP, check=True)
    esm_dir = OFFICIAL_CLEAN_APP / "esm"
    if not esm_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/facebookresearch/esm.git", str(esm_dir)], check=True)
    (OFFICIAL_CLEAN_APP / "data" / "esm_data").mkdir(parents=True, exist_ok=True)
    (OFFICIAL_CLEAN_APP / "data" / "distance_map").mkdir(parents=True, exist_ok=True)
    (OFFICIAL_CLEAN_APP / "data" / "model").mkdir(parents=True, exist_ok=True)
else:
    print("RUN_INSTALL_CLEAN is False; skipped official CLEAN clone/install.")

## Sync Prepared Tables Into Official CLEAN

This copies the normalized train/test tables into `official_CLEAN/app/data/`.

In [ ]:
def load_jobs() -> list[dict]:
    manifest_path = PREPARED_ROOT / "jobs_manifest.csv"
    if not manifest_path.exists():
        raise FileNotFoundError("Run the preparation cell first: " + str(manifest_path))
    return pd.read_csv(manifest_path).to_dict("records")


def sync_tables_to_clean_app() -> None:
    if not OFFICIAL_CLEAN_APP.exists():
        raise FileNotFoundError(f"Official CLEAN app not found: {OFFICIAL_CLEAN_APP}. Set RUN_INSTALL_CLEAN=True first or clone it manually.")
    data_dir = OFFICIAL_CLEAN_APP / "data"
    data_dir.mkdir(parents=True, exist_ok=True)
    jobs = load_jobs()
    names = set()
    for job in jobs:
        names.add(job["train_data"])
        names.add(job["test_data"])
    for name in sorted(names):
        src = PREPARED_ROOT / f"{name}.csv"
        dst = data_dir / f"{name}.csv"
        if not src.exists():
            raise FileNotFoundError(src)
        shutil.copy2(src, dst)
        print("copied", src, "->", dst)

if OFFICIAL_CLEAN_APP.exists():
    sync_tables_to_clean_app()
else:
    print("Official CLEAN app is not present yet; run install cell before syncing tables.")

## Generate ESM-1b Embeddings And Distance Maps

This is the first expensive step. It downloads/uses the ESM-1b model and writes per-protein embeddings to `official_CLEAN/app/data/esm_data/`.

For every training table, official CLEAN also mutates orphan EC sequences and computes distance maps.

In [ ]:
if RUN_GENERATE_ESM_AND_DISTANCES:
    if not OFFICIAL_CLEAN_APP.exists():
        raise FileNotFoundError(OFFICIAL_CLEAN_APP)
    sys.path.insert(0, str(OFFICIAL_CLEAN_APP / "src"))
    old_cwd = Path.cwd()
    os.chdir(OFFICIAL_CLEAN_APP)
    try:
        from CLEAN.utils import csv_to_fasta, retrive_esm1b_embedding, mutate_single_seq_ECs, compute_esm_distance, ensure_dirs

        ensure_dirs("data/esm_data")
        ensure_dirs("data/distance_map")
        jobs = load_jobs()
        train_names = sorted({job["train_data"] for job in jobs})
        test_names = sorted({job["test_data"] for job in jobs})
        all_names = sorted(set(train_names) | set(test_names))

        for name in all_names:
            print("[FASTA]", name)
            csv_to_fasta(f"data/{name}.csv", f"data/{name}.fasta")
            print("[ESM]", name)
            retrive_esm1b_embedding(name)

        for name in train_names:
            print("[MUTATE SINGLE-EC]", name)
            mutated_fasta_name = mutate_single_seq_ECs(name)
            print("[ESM MUTATED]", mutated_fasta_name)
            retrive_esm1b_embedding(mutated_fasta_name)
            print("[DISTANCE]", name)
            compute_esm_distance(name)
    finally:
        os.chdir(old_cwd)
else:
    print("RUN_GENERATE_ESM_AND_DISTANCES is False; skipped ESM/distance generation.")

## Train CLEAN Triplet Models

This runs the official `train-triplet.py` for each job's training set. The full CARE training baseline is expected to be much slower and larger than the metalloenzyme-only jobs.

In [ ]:
if RUN_TRAIN_CLEAN:
    jobs = load_jobs()
    for job in jobs:
        cmd = [
            PYTHON,
            "train-triplet.py",
            "--training_data", job["train_data"],
            "--model_name", job["model_name"],
            "--epoch", str(TRIPLET_EPOCHS),
            "--learning_rate", str(TRIPLET_LR),
        ]
        print("[TRAIN]", job["job_name"], " ".join(cmd))
        subprocess.run(cmd, cwd=OFFICIAL_CLEAN_APP, check=True)
else:
    print("RUN_TRAIN_CLEAN is False; skipped CLEAN training.")

## Run CLEAN Max-Separation Inference

Official CLEAN writes results as `official_CLEAN/app/results/<test_data>_maxsep.csv`. The notebook uses a run-specific duplicate test name for each job so results are not overwritten.

In [ ]:
if RUN_INFERENCE:
    jobs = load_jobs()
    for job in jobs:
        infer_code = (
            "from CLEAN.infer import infer_maxsep\n"
            f"infer_maxsep({job['train_data']!r}, {job['test_data']!r}, "
            f"report_metrics=True, pretrained=False, model_name={job['model_name']!r})\n"
        )
        print("[INFER]", job["job_name"])
        subprocess.run([PYTHON, "-c", infer_code], cwd=OFFICIAL_CLEAN_APP, check=True)
else:
    print("RUN_INFERENCE is False; skipped CLEAN inference.")

## Score EC1/EC2 Results

This scorer treats EC labels as multi-label sets. For each protein and EC level, the main top-1 metric is correct if CLEAN's first predicted EC prefix is in the true EC-prefix set.

It also reports multi-label macro F1 and macro recall using the predicted top-1 prefix set.

In [ ]:
def ec_prefix(ec: str, level: int) -> str | None:
    ec = str(ec).strip().replace("EC:", "")
    if not ec:
        return None
    parts = ec.split(".")
    if len(parts) < level:
        return None
    return ".".join(parts[:level])


def parse_clean_predictions(path: Path) -> dict[str, list[str]]:
    preds = {}
    with Path(path).open("r", encoding="utf-8", errors="ignore") as handle:
        reader = csv.reader(handle)
        for row in reader:
            if not row:
                continue
            entry = row[0].strip()
            ec_preds = []
            for item in row[1:]:
                item = item.strip()
                if not item.startswith("EC:"):
                    continue
                ec = item.split("/", 1)[0].replace("EC:", "").strip()
                if ec:
                    ec_preds.append(ec)
            preds[entry] = ec_preds
    return preds


def score_predictions(truth_csv: Path, prediction_csv: Path, *, levels=(1, 2)) -> pd.DataFrame:
    from sklearn.metrics import f1_score, recall_score
    from sklearn.preprocessing import MultiLabelBinarizer

    truth = pd.read_csv(truth_csv, sep="\t", dtype=str, keep_default_na=False)
    preds = parse_clean_predictions(prediction_csv)
    rows = []
    for level in levels:
        true_sets = []
        pred_top1_sets = []
        any_true_hits = []
        missing_predictions = 0
        for _, row in truth.iterrows():
            entry = row["Entry"]
            true_prefixes = {ec_prefix(ec, level) for ec in split_ecs(row["EC number"])}
            true_prefixes = {x for x in true_prefixes if x}
            pred_ecs = preds.get(entry, [])
            if not pred_ecs:
                missing_predictions += 1
                pred_prefixes = set()
            else:
                top1 = ec_prefix(pred_ecs[0], level)
                pred_prefixes = {top1} if top1 else set()
            true_sets.append(true_prefixes)
            pred_top1_sets.append(pred_prefixes)
            any_true_hits.append(bool(true_prefixes & pred_prefixes))
        labels = sorted(set().union(*true_sets, *pred_top1_sets))
        if labels:
            mlb = MultiLabelBinarizer(classes=labels)
            y_true = mlb.fit_transform(true_sets)
            y_pred = mlb.transform(pred_top1_sets)
            macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
            micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
            macro_recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
        else:
            macro_f1 = micro_f1 = macro_recall = 0.0
        rows.append({
            "ec_level": level,
            "n_test_proteins": int(len(truth)),
            "n_true_classes": int(len(set().union(*true_sets))) if true_sets else 0,
            "missing_predictions": int(missing_predictions),
            "top1_any_true_accuracy": float(sum(any_true_hits) / len(any_true_hits)) if any_true_hits else 0.0,
            "macro_f1_top1": float(macro_f1),
            "micro_f1_top1": float(micro_f1),
            "macro_recall_top1": float(macro_recall),
        })
    return pd.DataFrame(rows)


def score_all_jobs() -> pd.DataFrame:
    jobs = load_jobs()
    all_metrics = []
    for job in jobs:
        truth_csv = OFFICIAL_CLEAN_APP / "data" / f"{job['test_data']}.csv"
        pred_csv = OFFICIAL_CLEAN_APP / "results" / f"{job['test_data']}_maxsep.csv"
        if not pred_csv.exists():
            print("[SKIP missing prediction]", pred_csv)
            continue
        metrics = score_predictions(truth_csv, pred_csv)
        for key, value in job.items():
            metrics[key] = value
        out_dir = RESULTS_ROOT / job["job_name"]
        out_dir.mkdir(parents=True, exist_ok=True)
        metrics.to_csv(out_dir / "ec_level_metrics.csv", index=False)
        shutil.copy2(pred_csv, out_dir / pred_csv.name)
        all_metrics.append(metrics)
    if not all_metrics:
        return pd.DataFrame()
    combined = pd.concat(all_metrics, ignore_index=True)
    combined.to_csv(RESULTS_ROOT / "all_clean_predictor_metrics.csv", index=False)
    return combined

if RUN_SCORE_RESULTS:
    combined_metrics = score_all_jobs()
    display(combined_metrics)
else:
    print("RUN_SCORE_RESULTS is False; skipped scoring.")

## Expected Final Outputs

After table preparation, the important local files are:

```text
CLEAN/work/prepared_tables/jobs_manifest.csv
CLEAN/work/prepared_tables/table_summary.json
```

With the default selector, `jobs_manifest.csv` contains 12 jobs:

```text
5 CLEAN30 folds x 2 train scopes = 10 jobs
CARE30 x 2 train scopes = 2 jobs
```

After all long flags are enabled and completed, the important CLEAN outputs are:

```text
CLEAN/work/official_CLEAN/app/data/model/<model_name>.pth
CLEAN/work/official_CLEAN/app/results/<test_data>_maxsep.csv
CLEAN/work/scored_results/all_clean_predictor_metrics.csv
CLEAN/work/scored_results/<job_name>/ec_level_metrics.csv
```

Use `all_clean_predictor_metrics.csv` to compare `metallo` and `full` CLEAN training scopes against DeepMzyme on the same metalloenzyme-only test proteins. Do not compare against the original full CLEAN/CARE test sets here; this notebook intentionally tests only on the extracted metalloenzyme test subsets.
